<a href="https://colab.research.google.com/github/kircherlab/MPRAsnakeflow_tutorial/blob/main/tutorial_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !conda install -c conda-forge biopython -y
#!pip install matplotlib pandas snakemake
# !pip install numpy==1.23

In [2]:
!which python
!jupyter kernelspec list
!which apptainer
!apptainer --version
!conda config --set channel_priority strict
!conda config --show channel_priority

### working directory
%cd /ictstr01/groups/itg/teams/zeggini/projects/GO2/MPRA

~/miniconda3/envs/jupyter_env/bin/python
Available kernels:
  python3    /home/itg/oleg.vlasovets/miniconda3/envs/jupyter_env/share/jupyter/kernels/python3
/usr/bin/apptainer
apptainer version 1.4.2-1.el9
channel_priority: strict
/ictstr01/groups/itg/teams/zeggini/projects/GO2/MPRA


In [1]:
%%capture
import matplotlib.pyplot as plt
import os
import snakemake
import pandas
import gzip

from Bio import SeqIO

In [3]:
def runContainer(command):
    user_prompt = "\033[1;32muser@pc\033[0m"
    if docker:
        print(f"{user_prompt}$ docker run -v=${{PWD}}/mpra_test:/data/run --workdir /data/run {tutorial_container} {command}")
        !docker run -v=${{PWD}}/mpra_test:/data/run --workdir /data/run {tutorial_container} {command}
    elif apptainer:
        print(f"apptainer run -B=${{PWD}}/mpra_test:/data/run --cwd /data/run {tutorial_container} {command}")
        !apptainer run -B=${{PWD}}/mpra_test:/data/run --cwd /data/run mprasnakeflow_tutorial.sif {command}
    else:
        udocker(f"run -v=${{PWD}}/mpra_test:/data/run --workdir /data/run mprasnakeflow_tutorial {command}")

In [4]:
tutorial_container = "ovlasovets/mprasnakeflow_tutorial:0.5.4"
docker= False
apptainer = False

docker = !docker --help &>/dev/null; if [ $? -eq 0 ]; then echo 1; else echo 0; fi
docker = bool(int(docker[0]))
apptainer = !apptainer --help &>/dev/null; if [ $? -eq 0 ]; then echo 1; else echo 0; fi
apptainer = bool(int(apptainer[0]))

if docker:
    print("Docker is installed! We will use Docker to run MPRAsnakeflow")
elif apptainer:
    print("Apptainer is installed! We will use Apptainer to run MPRAsnakeflow")
else:
    print("Neither Docker nor Apptainer is installed. We assume you run the tutorial on Colab and we will install uDocker for Colab")

Apptainer is installed! We will use Apptainer to run MPRAsnakeflow


In [5]:
docker = False
apptainer = True

runContainer("snakemake --version")

!apptainer inspect --labels mprasnakeflow_tutorial.sif
!apptainer inspect --environment mprasnakeflow_tutorial.sif | grep -i version

apptainer run -B=${PWD}/mpra_test:/data/run --cwd /data/run ovlasovets/mprasnakeflow_tutorial:0.5.4 snakemake --version


9.3.5.dev0
distro_id: debian
org.label-schema.build-arch: amd64
org.label-schema.build-date: Monday_1_September_2025_19:8:22_UTC
org.label-schema.schema-version: 1.0
org.label-schema.usage.apptainer.version: 1.4.0
org.label-schema.usage.singularity.deffile.bootstrap: docker
org.label-schema.usage.singularity.deffile.from: ovlasovets/mprasnakeflow_tutorial:0.5.4
org.opencontainers.image.authors: Johannes Köster <johannes.koester@tu-dortmund.de>
org.opencontainers.image.created: 2025-05-05T18:59:32.171Z
org.opencontainers.image.description: Rapid builds of small Conda-based containers using micromamba.
org.opencontainers.image.licenses: Apache-2.0
org.opencontainers.image.revision: 74caf740c10a47e2509d252f49de3c752970cc1e
org.opencontainers.image.source: https://github.com/mamba-org/micromamba-docker
org.opencontainers.image.title: micromamba-docker
org.opencontainers.image.url: https://github.com/mamba-org/micromamba-docker
org.opencontainers.image.version: latest


In [ ]:
### barcode length 13
runContainer("snakemake --configfile config_assignment.yaml -c 10")

apptainer run -B=${PWD}/mpra_test:/data/run --cwd /data/run ovlasovets/mprasnakeflow_tutorial:0.5.4 snakemake --configfile config_assignment.yaml -c 10
Using profile mprasnakeflow for setting default command line arguments.
Running MPRAsnakeflow version 0.5.4
Assuming unrestricted shared filesystem usage.
host: cpusrv37.scidom.de
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 10
Rules claiming more threads will be scaled down.
Singularity containers: ignored
Job stats:
job                                            count
-------------------------------------------  -------
all                                                1
assignment_attach_idx                             60
assignment_check_design                            1
assignment_collect                                 1
assignment_collectBCs                              1
assignment_fastq_split                             3
assignment_filter                                  1
assignment_flagstat          

In [43]:
print("\nAverage length of forward reads:")
# Calculate the average length of all forward reads
barcode_length_forward = !zcat "/lustre/groups/itg/teams/zeggini/projects/GO2/MPRA/mpra_test/real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz" | awk 'NR % 4 == 2 {sum += length($0)} END {print sum*4/NR}'
barcode_length_forward = float(barcode_length_forward[0])
print(barcode_length_forward)

print("\nAverage length of reverse reads:")
# Calculate the average length of all reverse reads
barcode_length_reverse = !zcat "/lustre/groups/itg/teams/zeggini/projects/GO2/MPRA/mpra_test/real_data/assignment/reverse/24L005276_S11_L001_R2_001.fastq.gz" | awk 'NR % 4 == 2 {sum += length($0)} END {print sum*4/NR}'
barcode_length_reverse = float(barcode_length_reverse[0])
print(barcode_length_reverse)


Average length of forward reads:
151.0

Average length of reverse reads:
151.0


In [6]:
!conda create -n snakemake_env snakemake==8.20.5 -y

Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 25.7.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/itg/oleg.vlasovets/miniconda3/envs/snakemake_env

  added / updated specs:
    - snakemake==8.20.5


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    brotli-python-1.1.0        |  py312h1289d80_4         346 KB  conda-forge
    ca-certificates-2025.8.3   |       hbd8a1cb_0         151 KB  conda-forge
    certifi-2025.8.3           |     pyhd8ed1ab_0         155 KB  conda-forge
    cffi-1.17.1                |  py312h35888ee_1         288 KB  conda-forge
    charset-normalizer-3.4.3   |     pyhd8ed1ab_0          50 KB  conda-forge
    coin-or-cbc-2.10.12       

In [8]:
!conda init
!conda activate snakemake_env

no change     /home/itg/oleg.vlasovets/miniconda3/condabin/conda
no change     /home/itg/oleg.vlasovets/miniconda3/bin/conda
no change     /home/itg/oleg.vlasovets/miniconda3/bin/conda-env
no change     /home/itg/oleg.vlasovets/miniconda3/bin/activate
no change     /home/itg/oleg.vlasovets/miniconda3/bin/deactivate
no change     /home/itg/oleg.vlasovets/miniconda3/etc/profile.d/conda.sh
no change     /home/itg/oleg.vlasovets/miniconda3/etc/fish/conf.d/conda.fish
no change     /home/itg/oleg.vlasovets/miniconda3/shell/condabin/Conda.psm1
no change     /home/itg/oleg.vlasovets/miniconda3/shell/condabin/conda-hook.ps1
no change     /home/itg/oleg.vlasovets/miniconda3/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /home/itg/oleg.vlasovets/miniconda3/etc/profile.d/conda.csh
no change     /home/itg/oleg.vlasovets/.bashrc
No action taken.

CondaError: Run 'conda init' before 'conda activate'



In [58]:
print("\nLinker sequence used for barcode extraction:")
# The linker sequence is the constant region following the barcode in each read
linker = "GAATTCATCTGGTACCTCGGTTCACGCAATG"
print(linker)


Linker sequence used for barcode extraction:
GAATTCATCTGGTACCTCGGTTCACGCAATG


In [59]:
print("\nExample barcode and adapter sequences from forward reads:")
# Show the barcode sequence and the adapter sequence for the first 10 reads
!zcat "/lustre/groups/itg/teams/zeggini/projects/GO2/MPRA/mpra_test/real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz" \
| awk 'NR % 4 == 2 {match($0, /GAATTCATCTGGTACCTCGGTTCACGCAATG/); if (RSTART > 1) print substr($0, 1, RSTART-1) "\t" substr($0, RSTART, RLENGTH)}' | head



Example barcode and adapter sequences from forward reads:
NGGTCTGACATTC	GAATTCATCTGGTACCTCGGTTCACGCAATG
NGATAGGGTTACG	GAATTCATCTGGTACCTCGGTTCACGCAATG
GGCACTCCGGGCG	GAATTCATCTGGTACCTCGGTTCACGCAATG
GGTGTACCGTCAA	GAATTCATCTGGTACCTCGGTTCACGCAATG
GGCTACGCTTTGT	GAATTCATCTGGTACCTCGGTTCACGCAATG
GCAAAATTGCTCG	GAATTCATCTGGTACCTCGGTTCACGCAATG
GGGACGCATATCT	GAATTCATCTGGTACCTCGGTTCACGCAATG
GTCCGCTAAATGC	GAATTCATCTGGTACCTCGGTTCACGCAATG
GGCACGTAAGTTA	GAATTCATCTGGTACCTCGGTTCACGCAATG
GTTACGCTCTTCT	GAATTCATCTGGTACCTCGGTTCACGCAATG

gzip: stdout: Broken pipe


GAATTCATCTGGTACC
TCGGTTCACGCAATG

part of the linker is also in the designed files

In [9]:

!zcat "/lustre/groups/itg/teams/zeggini/projects/GO2/MPRA/mpra_test/real_data/archiv/bc_forward/DNA_forward/24L012064_S1_L001_R1_trimmed.fastq.gz" | head

@VL00287:57:AAG2FGCM5:1:1101:18799:1000 1:N:0:CGAGGCTA
TCGACGTTAGAG
+
C-C;CCC;-C-C
@VL00287:57:AAG2FGCM5:1:1101:19367:1000 1:N:0:CGAGGCTG
TTCTATCAACTG
+
;CCC;C-CCCCC
@VL00287:57:AAG2FGCM5:1:1101:19784:1000 1:N:0:CGAGGCTG
CGGGCCCACTAT

gzip: stdout: Broken pipe


In [11]:
!zcat "/lustre/groups/itg/teams/zeggini/projects/GO2/MPRA/mpra_test/real_data/assignment/reverse/24L005276_S11_L001_R2_001.fastq.gz" | head -20

@VL00287:48:AAFHK5NM5:1:1101:25313:1000 2:N:0:GCTACGCT
CCAGGACCGGATCAACTGCGTCGAACGCATTAATTTCTTTTTAGTCGAACCAGCCCGATTACCGGCGGGGAGTCTCACACATGCCTGATATTAGTATTTAGCCTGTACACTTCCGCGTCACTTTATAAAAAATCAGTGTGACTNTCATTAA
+
C-;CC;CCCCCCCCCCC;CCCCCC;CCC;-;CCCCC;CCCCCCCCCC;CCCCCCCCCCC-CC-CCCCCCCCC;CCCCCC;CC;CCCCCCCCCCCCCC-CCCCCCCC;CCCC-CCCC;C;-;C;C-C-;;C;CC-C;C-C;;CC#CC;CC-C
@VL00287:48:AAFHK5NM5:1:1101:25465:1000 2:N:0:GCTACGCT
CCAGGACCGGATCAACTTGTCCCTTGGAAGCCAGCTGCTTCCCCCTCCTGCTCATCCAGGAGGTAAACAGGACCGACAGCCAGCCTCTGCCACGTTCCCAGCCCCCTCCCCACCCAGCTCTGCGAGGTCCTGAATCCAGGAATNAACACCC
+
-CCCCCCCCCCCCCCCCCCCCCCCCCC;CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC;CC-CCCCCC-CCC-C;CCCC;C;CC;C;CCCCCCCCCCCCCCCCCCCCCCCC-CC;C;CC;--C-CCCC;CC;CC-;CC#C;CCC--
@VL00287:48:AAFHK5NM5:1:1101:26184:1000 2:N:0:GCTACGCT
CCAGGACCGGATCAACTGCGTTCCAAGGCAGGAAAGGCCACTTCAGAAGAGACTCTCTAACAAGGGACCAGCAGGGCAGGCTGGCTGACTGGTATCAGACCACCTTCAGGGCATCTCCTGACCTGAGCTGGGGCGAATTGCCTNAACCGGT
+
CC;CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC;CCCC

In [52]:
print("\n Internal control (4bp) sequences attached to barcode:")
!zcat "/lustre/groups/itg/teams/zeggini/projects/GO2/MPRA/mpra_test/real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz" \
| awk 'NR % 4 == 2 {match($0, /GAATTCATCTGGTACCTCGGTTCACGCAATG/); if (RSTART > 1) print substr($0, 1, 4)}' | head


 Internal control (4bp) sequences attached to barcode:
NGGT
NGAT
GGCA
GGTG
GGCT
GCAA
GGGA
GTCC
GGCA
GTTA

gzip: stdout: Broken pipe


In [ ]:
### 
runContainer("snakemake --configfile --sdm apptainer conda config_assignment.yaml -c 10")

apptainer run -B=${PWD}/mpra_test:/data/run --cwd /data/run ovlasovets/mprasnakeflow_tutorial:0.5.4 snakemake --configfile config_assignment.yaml -c 10
Using profile mprasnakeflow for setting default command line arguments.
Running MPRAsnakeflow version 0.5.4
Assuming unrestricted shared filesystem usage.
host: cpusrv37.scidom.de
Building DAG of jobs...
Creating conda environment /home/itg/oleg.vlasovets/.cache/snakemake/snakemake/source-cache/runtime-cache/tmpovl3qbue/file/data/MPRAsnakeflow/workflow/rules/../envs/cutadapt.yaml...
Traceback (most recent call last):

  File "/opt/conda/envs/snakemake/lib/python3.12/site-packages/snakemake/cli.py", line 2173, in args_to_api
    dag_api.execute_workflow(

  File "/opt/conda/envs/snakemake/lib/python3.12/site-packages/snakemake/api.py", line 602, in execute_workflow
    workflow.execute(

  File "/opt/conda/envs/snakemake/lib/python3.12/site-packages/snakemake/workflow.py", line 1266, in execute
    self.dag.create_conda_envs()

  File "/

In [21]:
# Step 2: Identify duplicate headers
headers = [record.id for record in records]
duplicates = [item for item, count in Counter(headers).items() if count > 1]

print(f"Found {len(duplicates)} duplicated headers:")
for dup in duplicates:
    print(dup)

# Step 3: Assign unique IDs to each replicate by appending _repX
output_unique_path = "real_data/assignment/designed_oligos_unique.fasta"
counter = defaultdict(int)

with open(output_unique_path, "w") as out:
    for record in records:
        counter[record.id] += 1
        record.id = f"{record.id}_rep{counter[record.id]}"
        record.description = record.id
        SeqIO.write(record, out, "fasta")

Found 1947 duplicated headers:
rs7310951_WT
rs1793955_WT
rs1250019_WT
rs1121980_WT
rs7206122_WT
rs7313483_WT
rs2482078_WT
rs2880977_WT
rs466233_WT
rs10453201_WT
rs6859171_WT
rs6877379_WT
rs7310579_WT
rs7201850_WT
rs1867449_WT
rs36715_WT
rs4761744_WT
rs61076166_WT
rs4912848_WT
rs2431531_WT
rs1558902_WT
rs36712_WT
rs6488715_WT
rs996733_WT
rs4979493_WT
rs12001098_WT
rs4237150_WT
rs1360291_WT
rs1552351_WT
rs944172_WT
rs6888752_WT
rs2445411_WT
rs2195272_WT
rs1421086_WT
rs4144502_WT
rs7187586_WT
rs2480933_WT
rs1337593_WT
rs1898673_WT
rs12533005_WT
rs3776870_WT
rs79460314_WT
rs459202_WT
rs6893547_WT
rs6538453_WT
rs455722_WT
rs460417_WT
rs6449762_WT
rs995358_WT
rs4449494_WT
rs10739688_WT
rs466404_WT
rs4129737_WT
rs2793007_WT
rs9923233_WT
rs1330360_WT
rs1113297_WT
rs1330356_WT
rs7023173_WT
rs10982535_WT
rs2001356_WT
rs7702524_WT
rs6861056_WT
rs4760615_WT
rs258236_WT
rs10819269_WT
rs4144501_WT
rs4293213_WT
rs60150206_WT
rs7959755_WT
rs60417486_WT
rs6477598_WT
rs28372466_WT
rs3751812_WT
rs9923312

In [22]:
# Step 4: Deduplicate based on sequence (remove redundant barcodes)
seen_seqs = {}
unique_records = []

for record in SeqIO.parse(output_unique_path, "fasta"):
    seq = str(record.seq)
    if seq not in seen_seqs:
        seen_seqs[seq] = record.id
        unique_records.append(record)
    else:
        print(f"Collision: {record.id} and {seen_seqs[seq]}")

# Step 5: Save deduplicated records
output_dedup_path = "real_data/assignment/designed_oligos_deduplicated.fasta"
with open(output_dedup_path, "w") as out:
    SeqIO.write(unique_records, out, "fasta")

Collision: scr_153_rep1 and scr_38_rep1
Collision: scr_344_rep1 and scr_115_rep1
Collision: scr_356_rep1 and scr_274_rep1
Collision: scr_402_rep1 and scr_352_rep1
Collision: scr_496_rep1 and scr_26_rep1
Collision: scr_153_rep2 and scr_38_rep2
Collision: scr_344_rep2 and scr_115_rep2
Collision: scr_356_rep2 and scr_274_rep2
Collision: scr_402_rep2 and scr_352_rep2
Collision: scr_496_rep2 and scr_26_rep2
Collision: scr_153_rep3 and scr_38_rep3
Collision: scr_344_rep3 and scr_115_rep3
Collision: scr_356_rep3 and scr_274_rep3
Collision: scr_402_rep3 and scr_352_rep3
Collision: scr_496_rep3 and scr_26_rep3
Collision: scr_153_rep4 and scr_38_rep4
Collision: scr_344_rep4 and scr_115_rep4
Collision: scr_356_rep4 and scr_274_rep4
Collision: scr_402_rep4 and scr_352_rep4
Collision: scr_496_rep4 and scr_26_rep4


There are 500 generated scrambled sequences (e.g., scr_1 to scr_500) as negative controls, likely by:
Randomly permuting real CRE sequences, but randomness is not a guarantee of uniqueness, so these collision are explainable

In [23]:
### check length of oligos in resulted designed oligos file
!awk '/^>/ {if (seq) print length(seq); seq=""; next} {seq=seq $0} END {if (seq) print length(seq)}' real_data/assignment/designed_oligos_deduplicated.fasta | sort -nu

300


In [24]:
### check number of IDs in the resulted designed oligos file
seq_dict = defaultdict(set)

for record in SeqIO.parse("real_data/assignment/designed_oligos_deduplicated.fasta", "fasta"):
    base_id = record.id.split("_rep")[0]
    seq_dict[base_id].add(str(record.seq))

print(f"Number of unique base IDs: {len(seq_dict)}")

seen_seqs = {}
unique_records = []
collision_count = 0

for record in SeqIO.parse("real_data/assignment/designed_oligos_deduplicated.fasta", "fasta"):
    seq = str(record.seq)
    if seq not in seen_seqs:
        seen_seqs[seq] = record.id
        unique_records.append(record)
    else:
        print(f"Collision: {record.id} and {seen_seqs[seq]}")
        collision_count += 1

# Summary output
if collision_count == 0:
    print("✅ No collisions found — all sequences are unique.")
else:
    print(f"⚠️ {collision_count} collisions found among sequences.")

Number of unique base IDs: 1942
✅ No collisions found — all sequences are unique.


### Investigate barcode length

In [25]:
import gzip

# Path to DNA R1 file (barcode should be here)
dna_r1_path = "real_data/bc_forward/DNA_forward/24L012064_S1_L001_R1_trimmed.fastq.gz"

# Read first 10 reads
with gzip.open(dna_r1_path, "rt") as handle:
    for i, record in enumerate(SeqIO.parse(handle, "fastq")):
        print(f"Read {i+1}:")
        print(str(record.seq))
        print("-" * 40)
        if i == 9:
            break

Read 1:
TCGACGTTAGAG
----------------------------------------
Read 2:
TTCTATCAACTG
----------------------------------------
Read 3:
CGGGCCCACTAT
----------------------------------------
Read 4:
GGAAGTTCGGTT
----------------------------------------
Read 5:
CCACGCTAGTTG
----------------------------------------
Read 6:
ACTTCGGGTAGT
----------------------------------------
Read 7:
AATTGGTCTGTC
----------------------------------------
Read 8:
CAGACGGGACGA
----------------------------------------
Read 9:
AGGACGTTGGCT
----------------------------------------
Read 10:
ATTACTACGCTT
----------------------------------------


Interpretation
These reads are already trimmed (_trimmed.fastq.gz) — likely by a tool like cutadapt or Trimmomatic.

The trimming step has likely clipped adapters and isolated the barcode region, so the read contains only the barcode.

Since all reads are 12 bp, we can reasonably conclude:

🔑 Your barcode length is 12 nucleotides (bc_length: 12, also alligns with Faye's comments) 

### Investigate linker sequences length

In [21]:
r1_path = "real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz"
barcode_length = 12
linker_length = 30  # can adjust to test

linker_seqs = []

with gzip.open(r1_path, "rt") as handle:
    for i, record in enumerate(SeqIO.parse(handle, "fastq")):
        seq = str(record.seq)
        if len(seq) >= barcode_length + linker_length:
            linker = seq[barcode_length:barcode_length + linker_length]
            linker_seqs.append(linker)
        if i >= 10000:
            break

# Count most common linker sequences
counter = Counter(linker_seqs)
print("Most common linker sequences (top 5):")
for linker, count in counter.most_common(5):
    print(f"{linker} → {count} times")

Most common linker sequences (top 5):
GGAATTCATCTGGTACCTCGGTTCACGCAA → 2608 times
TGAATTCATCTGGTACCTCGGTTCACGCAA → 2569 times
AGAATTCATCTGGTACCTCGGTTCACGCAA → 2135 times
CGAATTCATCTGGTACCTCGGTTCACGCAA → 2003 times
GAATTCATCTGGTACCTCGGTTCACGCAAT → 45 times


In [24]:
import gzip
from itertools import islice

fastq_path = "real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz"

lengths = []
with gzip.open(fastq_path, "rt") as f:
    # FASTQ format: every 4 lines, 2nd line is the sequence
    for i, line in enumerate(islice(f, 0, 4000)):  # first 1000 records = 4000 lines
        if i % 4 == 1:  # sequence line
            lengths.append(len(line.strip()))

# unique, sorted
unique_lengths = sorted(set(lengths))

# take last 10
print(unique_lengths[-10:])

[151]


In [25]:
import gzip
from collections import Counter
from itertools import islice

# --- CONFIG ---
R1_PATH = "real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz"
R2_PATH = "real_data/assignment/reverse/24L005276_S11_L001_R2_001.fastq.gz"  # change if needed
MOTIF_R2 = "CCAGGACCGGATCAACT"  # constant anchor you grepped in R2
LINKER_R1 = "GAATTCATCTGGTACCTCGGTTCACGCAATG"  # expected linker on R1
N_HEAD_RECORDS = 1000  # how many records to sample for quick checks (set to None for full)
SHOW_EXAMPLES = 5      # how many example lines to print

def iter_fastq_sequences(gz_path, max_records=None):
    """Yield sequence lines (2nd of each 4-line FASTQ record)."""
    with gzip.open(gz_path, "rt") as f:
        if max_records is not None:
            # Read only first max_records*4 lines
            it = islice(f, 0, max_records * 4)
        else:
            it = f
        rec = []
        for i, line in enumerate(it, 1):
            rec.append(line.rstrip("\n"))
            if len(rec) == 4:
                yield rec[1]  # sequence line
                rec = []

def summarize_lengths(gz_path, max_records=N_HEAD_RECORDS):
    lengths = Counter(len(seq) for seq in iter_fastq_sequences(gz_path, max_records))
    total = sum(lengths.values())
    unique = sorted(lengths)
    print(f"[{gz_path}] sampled {total} reads")
    print(f"  unique lengths (sorted): {unique[-10:] if len(unique)>10 else unique}")
    # show top 3 most common lengths
    print("  top lengths:", lengths.most_common(3))
    return lengths

def check_r2_motif_at_start(gz_path, motif, max_records=N_HEAD_RECORDS):
    count = 0
    examples = []
    for seq in iter_fastq_sequences(gz_path, max_records):
        if seq.startswith(motif):
            count += 1
            if len(examples) < SHOW_EXAMPLES:
                examples.append(seq[:len(motif)+30])  # show motif + a bit more
    print(f"[{gz_path}] sequences starting with motif '{motif}': {count}")
    if examples:
        print("  examples:")
        for ex in examples:
            print("   ", ex)
    return count, examples

def find_linker_positions(gz_path, linker, max_records=N_HEAD_RECORDS):
    pos_counter = Counter()
    examples = []
    for seq in iter_fastq_sequences(gz_path, max_records):
        idx = seq.find(linker)
        if idx != -1:
            pos_counter[idx] += 1
            if len(examples) < SHOW_EXAMPLES:
                span = seq[idx: idx + len(linker)]
                context = seq[max(0, idx-10): idx] + "[" + span + "]" + seq[idx+len(linker): idx+len(linker)+10]
                examples.append((idx, context))
    total_hits = sum(pos_counter.values())
    print(f"[{gz_path}] linker '{linker}' found in {total_hits} reads")
    if total_hits:
        top_pos = pos_counter.most_common(5)
        print("  top positions (pos, count):", top_pos)
        print("  examples (pos, context):")
        for pos, ctx in examples:
            print(f"   pos={pos}  ...{ctx}...")
    return pos_counter, examples

if __name__ == "__main__":
    print("=== LENGTH SUMMARIES ===")
    summarize_lengths(R1_PATH)
    summarize_lengths(R2_PATH)

    print("\n=== R2 MOTIF AT START ===")
    check_r2_motif_at_start(R2_PATH, MOTIF_R2)

    print("\n=== R1 LINKER POSITIONS ===")
    find_linker_positions(R1_PATH, LINKER_R1)

=== LENGTH SUMMARIES ===
[real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz] sampled 1000 reads
  unique lengths (sorted): [151]
  top lengths: [(151, 1000)]
[real_data/assignment/reverse/24L005276_S11_L001_R2_001.fastq.gz] sampled 1000 reads
  unique lengths (sorted): [151]
  top lengths: [(151, 1000)]

=== R2 MOTIF AT START ===
[real_data/assignment/reverse/24L005276_S11_L001_R2_001.fastq.gz] sequences starting with motif 'CCAGGACCGGATCAACT': 966
  examples:
    CCAGGACCGGATCAACTGCGTCGAACGCATTAATTTCTTTTTAGTCG
    CCAGGACCGGATCAACTTGTCCCTTGGAAGCCAGCTGCTTCCCCCTC
    CCAGGACCGGATCAACTGCGTTCCAAGGCAGGAAAGGCCACTTCAGA
    CCAGGACCGGATCAACTGCGTGGCAGGGACAACTAAGGGAATAAAAG
    CCAGGACCGGATCAACTGAGGTGTCCAGGTGGGGCTCCTATTCTTTG

=== R1 LINKER POSITIONS ===
[real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz] linker 'GAATTCATCTGGTACCTCGGTTCACGCAATG' found in 937 reads
  top positions (pos, count): [(13, 933), (12, 2), (3, 1), (14, 1)]
  examples (pos, context):
   pos=13  

In [26]:
import gzip
from collections import Counter

R1 = "real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz"
LINKER = "GAATTCATCTGGTACCTCGGTTCACGCAATG"
BC_LEN = 16

def seqs(gz):
    with gzip.open(gz, "rt") as f:
        while True:
            hdr = f.readline()
            if not hdr: break
            s   = f.readline().strip()
            f.readline(); f.readline()
            yield s

n = hits = ok = 0
pos_ct = Counter()
bc_ct = Counter()

for s in seqs(R1):
    n += 1
    i = s.find(LINKER)
    if i != -1:
        hits += 1
        pos_ct[i] += 1
        start = i + len(LINKER)
        if start + BC_LEN <= len(s):
            bc = s[start:start+BC_LEN]
            bc_ct[bc] += 1
            ok += 1

print(f"Total reads: {n}")
print(f"Reads with linker: {hits} ({hits/n:.1%})")
print("Top linker positions:", pos_ct.most_common(5))
print(f"Barcodes extracted: {ok} ({ok/n:.1%} of all reads)")
print("Top barcodes:", bc_ct.most_common(10))


Total reads: 8449233
Reads with linker: 7963791 (94.3%)
Top linker positions: [(13, 7912378), (12, 38227), (14, 6487), (11, 2940), (10, 1290)]
Barcodes extracted: 7963791 (94.3% of all reads)
Top barcodes: [('TCCCCAACTGCACCAA', 80683), ('CCAGAGGAAGCCACGG', 79299), ('GCCATAGGGGCCAGTG', 72784), ('CCCACAGCTCCAGGAG', 71954), ('CCCTGCCCTCATGACT', 65203), ('GCCGGACCGGGGGGCT', 61466), ('AGAGGCTGGAAGCCAA', 61128), ('GCTCTAAGCAGCAGCT', 58702), ('CCAAGTCGGGTCGGGC', 57813), ('CCCTACCTAAATCATA', 56887)]


In [27]:
import gzip
from collections import Counter

R1 = "real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz"
LINKER = "GAATTCATCTGGTACCTCGGTTCACGCAATG"  # your linker sequence
BC_LEN = 16  # expected barcode length

def seqs(gz):
    with gzip.open(gz, "rt") as f:
        while True:
            hdr = f.readline()
            if not hdr: break
            s   = f.readline().strip()
            f.readline(); f.readline()
            yield s

n = hits = 0
lengths = Counter()
examples = []

for s in seqs(R1):
    n += 1
    i = s.find(LINKER)
    if i != -1:
        hits += 1
        start = i + len(LINKER)
        bc = s[start:start+BC_LEN]  # take 16 bases after linker
        lengths[len(bc)] += 1
        if len(examples) < 5:
            examples.append(bc)

print(f"Total reads scanned: {n}")
print(f"Reads with linker: {hits} ({hits/n:.1%})")
print("Barcode length distribution:", lengths)
print("Example barcodes:", examples)


Total reads scanned: 8449233
Reads with linker: 7963791 (94.3%)
Barcode length distribution: Counter({16: 7963791})
Example barcodes: ['GTGTCGACTGTACAGG', 'CCCACAGCTCCAGGAG', 'GCAAGATGCCTGACTC', 'CAGACTGTTTTCCACC', 'GAGGAGCTACGGTAGG']


✅ Why Test Exactly 18–20?
1. Design Standards from MPRA Protocols
Many published MPRA designs (e.g., from Tewhey, Kircher, or Ulirsch labs) use linkers between 18 and 20 bp, often including restriction sites like EcoRI and KpnI.

2. This known biological prior gives you a strong expectation that the linker is ~19 bp.

### Create subsets for testing the pipeline

In [6]:
# Define input and output paths
IN_R1="/home/itg/oleg.vlasovets/MPRA/MPRAsnakeflow_tutorial/real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz"
IN_R2="/home/itg/oleg.vlasovets/MPRA/MPRAsnakeflow_tutorial/real_data/assignment/reverse/24L005276_S11_L001_R2_001.fastq.gz"

OUT_R1="real_data/assignment/forward/24L005276_S11_L001_R1_001_subset.fastq.gz"
OUT_R2="real_data/assignment/reverse/24L005276_S11_L001_R2_001_subset.fastq.gz"

# Extract first 100,000 reads = 400,000 lines
!zcat "$IN_R1" | head -n 400000 | gzip > "$OUT_R1"
!zcat "$IN_R2" | head -n 400000 | gzip > "$OUT_R2"



gzip: stdout: Broken pipe

gzip: stdout: Broken pipe


#### Review this block

In [ ]:
from Bio import SeqIO

for i, rec in enumerate(SeqIO.parse("real_data/assignment/designed_oligos_deduplicated.fasta", "fasta")):
    print(f"Design {i}: {str(rec.seq)[:50]}")
    if i == 10:
        break

Design 0: AGGACCGGATCAACTTCAGCACACCTCTCCCTCAGGGAGAGATCAGAACT
Design 1: AGGACCGGATCAACTTCAGTAGGACATCTGTATGAATGCTGGGAAGGGGT
Design 2: AGGACCGGATCAACTTCAGGGCTCATGGTGGTCTGAGGGCAGTGGGTGTT
Design 3: AGGACCGGATCAACTTCAGCTCCAACTCATATTGCATTTTCTGTCCAGCA
Design 4: AGGACCGGATCAACTTCAGAGTACGAGACCAGCCTGGCCAACATGGTGAA
Design 5: AGGACCGGATCAACTTCAGTTAACCAAGTTGGAAGGGACTGAAGCAATGA
Design 6: AGGACCGGATCAACTTCAGGGCCCTAATGTAATTAGCTCTCCTGTAATCT
Design 7: AGGACCGGATCAACTTCAGATAAGAAGATTCACAGAAAATACTTAGCACG
Design 8: AGGACCGGATCAACTTCAGGTAAAATGGATTTGGGTCGTAAGCTTAGCTC
Design 9: AGGACCGGATCAACTTCAGACTGGCCTGTTTTATCCCAAGCTAGTTACAT
Design 10: AGGACCGGATCAACTTCAGTTTCATTGAGAAATTTGAAACAAAAAAGTGA


In [ ]:
from Bio.Seq import Seq

design_rc = set(str(Seq(record.seq).reverse_complement())[:25]
                for record in SeqIO.parse(design_path, "fasta"))

# Sample R1 reads again
with gzip.open(r1_path, "rt") as f:
    for i, rec in enumerate(SeqIO.parse(f, "fastq")):
        seq = str(rec.seq)
        for oligo in design_rc:
            if oligo in seq:
                print(f"✅ Match found in Read {i}: {oligo}")
                break
        if i > 20:
            break

✅ Match found in Read 0: TCGGTTCACGCAATGGTGTCGACTG
✅ Match found in Read 1: TCGGTTCACGCAATGCCCACAGCTC
✅ Match found in Read 2: TCGGTTCACGCAATGGCAAGATGCC
✅ Match found in Read 3: TCGGTTCACGCAATGCAGACTGTTT
✅ Match found in Read 4: TCGGTTCACGCAATGGAGGAGCTAC
✅ Match found in Read 5: TCGGTTCACGCAATGCCAAGTCGGG
✅ Match found in Read 6: TCGGTTCACGCAATGAAGAATGTGT
✅ Match found in Read 7: TCGGTTCACGCAATGCGGGAGATGT
✅ Match found in Read 8: TCGGTTCACGCAATGCCACTTCCTC
✅ Match found in Read 9: TCGGTTCACGCAATGGGTCTTGCGG
✅ Match found in Read 10: TCGGTTCACGCAATGGCTAAGGATT
✅ Match found in Read 11: TCGGTTCACGCAATGGGACCTGAAA
✅ Match found in Read 12: TCGGTTCACGCAATGAAGGGGACTT
✅ Match found in Read 13: TCGGTTCACGCAATGCCTTTGAAGG
✅ Match found in Read 14: TCGGTTCACGCAATGGCGTCAGGAT
✅ Match found in Read 15: TCGGTTCACGCAATGATAAGCAGAA
✅ Match found in Read 16: TCGGTTCACGCAATGAGGCCAAGGT
✅ Match found in Read 17: TCGGTTCACGCAATGGTTCACTGCG
✅ Match found in Read 18: TCGGTTCACGCAATGGACCTTGGGA
✅ Match found in Read 

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq

with open("real_data/assignment/designed_oligos_deduplicated_rc.fasta", "w") as out_f:
    for record in SeqIO.parse("real_data/assignment/designed_oligos_deduplicated.fasta", "fasta"):
        rc_seq = str(Seq(record.seq).reverse_complement())
        record.seq = Seq(rc_seq)
        SeqIO.write(record, out_f, "fasta")


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq

input_path = "real_data/assignment/designed_oligos_deduplicated.fasta"
output_path = "real_data/assignment/designed_oligos_rc.fasta"

with open(output_path, "w") as out_f:
    for record in SeqIO.parse(input_path, "fasta"):
        rc_seq = record.seq.reverse_complement()
        record.seq = rc_seq
        SeqIO.write(record, out_f, "fasta")

In [ ]:
from Bio import SeqIO
lengths = [len(rec.seq) for rec in SeqIO.parse("real_data/assignment/designed_oligos_rc.fasta", "fasta")]
print(min(lengths), max(lengths))

300 300


In [ ]:
from Bio.Seq import Seq
from Bio import SeqIO

design_path = "real_data/assignment/designed_oligos_deduplicated.fasta"
rc_oligos = {str(Seq(record.seq).reverse_complement())[:25] for record in SeqIO.parse(design_path, "fasta")}


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
import gzip

# Load reverse-complemented oligos
rc_oligos = {str(record.seq) for record in SeqIO.parse("real_data/assignment/designed_oligos_rc.fasta", "fasta")}

# Check trimmed R1 sequences
barcode_len = 12
linker_len = 19
start = barcode_len + linker_len

r1_path = "real_data/assignment/forward/24L005276_S11_L001_R1_001_subset.fastq.gz"
with gzip.open(r1_path, "rt") as handle:
    for i, record in enumerate(SeqIO.parse(handle, "fastq")):
        read_seq = str(record.seq)
        trimmed = read_seq[start:]
        if trimmed in rc_oligos:
            print(f"✅ Match found in read {i}")
            break
        if i >= 10000:
            print("❌ No exact matches found in first 10,000 reads.")
            break


❌ No exact matches found in first 10,000 reads.


In [ ]:
!zcat results/assignment/assignMPRAworkshop/barcodes_incl_other.tsv.gz | head


AAGCAATAGGTGTG	other	NA
AATATTGGTAGCGG	other	NA
AATATTGTCCCACG	other	NA
ACATTTGTTATATG	other	NA
ACCGAGTAGTGGCG	other	NA
ACCTTTTTACATTG	other	NA
ACTGTGGTGTTTAG	other	NA
AGACTCAGAGTGTG	other	NA
AGCATCCTAGAACG	other	NA
AGCGGTAACGTATG	other	NA

gzip: stdout: Broken pipe


In [ ]:
from collections import Counter
from Bio import SeqIO
import gzip

r1_path = "real_data/assignment/forward/24L005276_S11_L001_R1_001_subset.fastq.gz"
barcode_length = 12

for linker_length in [18, 19, 20, 21, 22, 23, 24, 25, 30, 33]:
    linker_seqs = []
    with gzip.open(r1_path, "rt") as handle:
        for i, record in enumerate(SeqIO.parse(handle, "fastq")):
            seq = str(record.seq)
            if len(seq) >= barcode_length + linker_length:
                linker = seq[barcode_length:barcode_length + linker_length]
                linker_seqs.append(linker)
            if i >= 10000:
                break
    counter = Counter(linker_seqs)
    top = counter.most_common(1)[0] if counter else ("-", 0)
    print(f"Linker length {linker_length}: {top[0]} → {top[1]} reads")


Linker length 18: GGAATTCATCTGGTACCT → 2668 reads
Linker length 19: GGAATTCATCTGGTACCTC → 2668 reads
Linker length 20: GGAATTCATCTGGTACCTCG → 2649 reads
Linker length 21: GGAATTCATCTGGTACCTCGG → 2639 reads
Linker length 22: GGAATTCATCTGGTACCTCGGT → 2638 reads
Linker length 23: GGAATTCATCTGGTACCTCGGTT → 2636 reads
Linker length 24: GGAATTCATCTGGTACCTCGGTTC → 2634 reads
Linker length 25: GGAATTCATCTGGTACCTCGGTTCA → 2631 reads
Linker length 30: GGAATTCATCTGGTACCTCGGTTCACGCAA → 2608 reads
Linker length 33: GGAATTCATCTGGTACCTCGGTTCACGCAATGC → 816 reads


In [ ]:
!zcat real_data/assignment/forward/24L005276_S11_L001_R1_001.fastq.gz | awk 'END {print NR/4}'


8449233


In [ ]:
### Reverse complement of an oligo
from Bio.Seq import Seq
oligo = "GGAAGGGGTCACCTGCACATATATCATCCACCAGCACTGGA"
print(Seq(oligo).reverse_complement())

TCCAGTGCTGGTGGATGATATATGTGCAGGTGACCCCTTCC


#### Run pipeline

In [24]:
!apptainer exec --bind /home/itg/oleg.vlasovets:/MPRAsnakeflow /home/itg/oleg.vlasovets/mprasnakeflow_tutorial.sif python --version

Python 3.12.10


In [13]:
!cat results/assignment/assignMPRAworkshop/design_check.err

## Assignment results

Let's take a look at the files in the assignment results folder!

In [14]:
results_dir = "results/assignment"
!ls -R "{results_dir}"

results/assignment:
assignMPRAworkshop

results/assignment/assignMPRAworkshop:
aligned_merged_reads.bam      design_check.done  reference
aligned_merged_reads.bam.bai  design_check.err	 statistic
barcodes_incl_other.tsv.gz    fastq

results/assignment/assignMPRAworkshop/fastq:
BC.byLength.fastq.gz  FW.byLength.fastq.gz

results/assignment/assignMPRAworkshop/reference:
reference.fa

results/assignment/assignMPRAworkshop/statistic:
assignment  total_counts.tsv

results/assignment/assignMPRAworkshop/statistic/assignment:
bam_stats.txt


The statistic folder contains run statistics for the assignment step, such as the total number of associated barcodes, the number of oligos with at least 15 barcodes, and more. We will now just take a look at the generated count histogram of barcodes per oligo.

The median number of barcodes is 24. We usually require each tested sequence to have at least 10 barcodes associated.

We will now take a look at the actual output file of the assignment run, which is a sorted table of `barcodes`, their assigned `sequence IDs`, information about the `quality` of the alignment (fwd/rev reads to designed oligos), and how often this barcode was assigned to this sequence compared to the total number this barcode was observed in the experiment. The information in the quality field is derived from the SAM/BAM output (https://samtools.github.io/hts-specs/SAMv1.pdf) and represents a concatenation of the SAM position, cigar, NM, AM, and mapQ.

In [25]:
assignment = os.path.join(results_dir, "assignMPRAworkshop/assignment_barcodes.default.tsv.gz")
!zcat "{assignment}" | awk 'NR <= 10 {{print $$0}}'

gzip: results/assignment/assignMPRAworkshop/assignment_barcodes.default.tsv.gz: No such file or directory


We would like to know how many different oligos have been included in the final assignment.\
When we compare this number to the number of sequences in our design, we can get a feeling of how many sequences were able to get assigned with their barcodes. This has implications for the proportion of targets that we can eventually analyse and whether each target is supported with sufficiently many barcodes. This information is important for the experiment overall and analyzing the counts of each individual replicate.

In [52]:
assigned_sequences = !zcat "{assignment}" | awk -F'\t' '{{names[$$2]++}} END {{print length(names)}}'

assigned_sequences = int(assigned_sequences[0])
print("Number of sequences we were able to assign: %s"%(assigned_sequences))

designed_sequences = !cat "example_data/design/workshop_design.fa" | grep -v ">" | wc -l
designed_sequences = int(designed_sequences[0])
print("Number of sequences in the initial design: %s"%(designed_sequences))

print(r"Conclusion: %s%% sequences of our initial design can be used in the experimental/counting step"%(round(assigned_sequences/designed_sequences * 100, 1)))

Number of sequences we were able to assign: 5
Number of sequences in the initial design: 2104
Conclusion: 0.2% sequences of our initial design can be used in the experimental/counting step


### QC report

We have a QC report for the assignment as well as for the experiment workflow. For our run you can find it here:

[MPRAsnakeflow_tutorial/results/assignment/assignMPRAworkshop/qc_report.default.html](MPRAsnakeflow_tutorial/results/assignment/assignMPRAworkshop/qc_report.default.html)

The QC report gives a nice overview of the assignment run and might be the first thing you will look at.

In [ ]:
from IPython.display import IFrame

IFrame(src='./MPRAsnakeflow_tutorial/results/assignment/assignMPRAworkshop/qc_report.default.html', width=700, height=600)